In [3]:
!pip install -q langchain langchain-community langchain-core chromadb sentence-transformers pypdf python-docx pandas groq openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "YOUR_API_KEY"

In [5]:
from google.colab import files
import pandas as pd
from docx import Document
from langchain_community.document_loaders import PyPDFLoader

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

text = ""

if file_name.endswith(".pdf"):
    loader = PyPDFLoader(file_name)
    documents = loader.load()
    text = " ".join([doc.page_content for doc in documents])

elif file_name.endswith(".docx"):
    doc = Document(file_name)
    text = " ".join([para.text for para in doc.paragraphs])

elif file_name.endswith(".csv"):
    df = pd.read_csv(file_name)
    text = df.to_string()

elif file_name.endswith(".xlsx"):
    df = pd.read_excel(file_name)
    text = df.to_string()

print("Text extraction completed ✅")

Saving Ajay-Resume.pdf to Ajay-Resume.pdf
Text extraction completed ✅


In [6]:
import sys
!{sys.executable} -m pip install langchain-community

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

docs = splitter.create_documents([text])

In [33]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os

# Ensure directory exists
if not os.path.exists('./chroma_db'):
    os.makedirs('./chroma_db')

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Initialize Chroma with persist_directory
vectorstore = Chroma.from_documents(
    docs,
    embedding,
    persist_directory="./chroma_db"
)
retriever = vectorstore.as_retriever()
print("Vector store initialized and persisted to ./chroma_db ✅")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store initialized and persisted to ./chroma_db ✅


In [9]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

def ask_llm(question, context):
    prompt = f"""
    Based ONLY on the following context, answer the question below.
    If the answer cannot be found in the context, please state that clearly.

    Context:
    {context}

    Question: {question}
    Answer:"""

    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )

    return completion.choices[0].message.content

In [10]:
query = input("What would you like to ask about the document? ")
relevant_docs = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in relevant_docs])

answer = ask_llm(query, context)
print(answer)

What would you like to ask about the document? what are the projects are done by me?
Based on the provided context, the projects done by you are:

1. Help Desk Ticketing System (using Spring Boot, React.js, and MySQL)
2. Swiggy Sentiment Analysis (using ML, Classification technique, and NLP)
3. Document Chatbot (using Python, LangChain, ChromaDB, Sentence Transformers, and Pandas)

These projects are mentioned in the context under the section that describes your experiences and achievements.


In [34]:
import shutil
import os
from google.colab import files

if os.path.exists('./chroma_db'):
    # Zip the chroma_db directory
    shutil.make_archive('chroma_db_backup', 'zip', './chroma_db')
    # Download the zipped database
    files.download('chroma_db_backup.zip')
    print('Vector store backup created and download started! ✅')
else:
    print('Error: ./chroma_db directory not found. Please run the vector store creation cell first.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Vector store backup created and download started! ✅


### Step 1: Install Dependencies
We begin by installing the necessary libraries for LangChain, document processing, and vector storage.

### Step 2: Configure API Keys
Set up your Groq API key as an environment variable to allow the LLM to process queries.

### Step 3: Document Loading & Extraction
Upload your files (PDF, DOCX, CSV, XLSX) and extract the raw text content for processing.

### Step 4: Text Splitting
Divide the large extracted text into smaller, overlapping chunks to ensure the LLM stays within its context window.

### Step 5: Vector Store Initialization
Convert text chunks into numerical embeddings using HuggingFace and store them in a Chroma vector database for fast retrieval.

### Step 6: Querying the LLM
Define the retrieval logic and prompt the LLM to answer questions based strictly on the retrieved document context.

### Step 7: Backup and Download
Compress the persistent vector database into a ZIP file and download it for local use or future sessions.